### Question 1
Download a dataset of mobile phone prices (with features like RAM, storage, camera MP, battery, brand, etc.) and fit a multiple linear regression model to predict the price using at least three features of your choice.

Multiple Linear Regression models the relationship between multiple explanatory variables and a continuous response variable using the equation $\hat{y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_k X_k$. Here, four hardware features (`ram_gb`, `storage_gb`, `camera_mp`, and `battery_mah`) are selected to predict the target price (`price_inr`).

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Load the mobile phone prices dataset
df = pd.read_csv('mobile_phone_prices_500.csv')

# Define selected features and target
features = ['ram_gb', 'storage_gb', 'camera_mp', 'battery_mah']
target = 'price_inr'

X = df[features]
y = df[target]

# Split data into 80% train and 20% test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit multiple linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Model fitted successfully.")
print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")
display(df.head())

Model fitted successfully.
Training samples: 400 | Testing samples: 100


,ram_gb,storage_gb,camera_mp,battery_mah,price_inr
0,6,96,12,4000,31800.0
1,16,272,108,4000,77000.0
2,8,112,50,5500,42770.0
3,8,112,48,5500,43020.0
4,4,48,12,5500,27380.0


### Question 2
Display the regression coefficients and intercept from your model, and write a short comment on what each coefficient means in the context of mobile phones (e.g., how much price increases per extra GB of RAM).

The regression coefficients quantify the expected change in phone price (in INR) associated with a one-unit increase in each feature, holding all other variables constant:

- **Intercept ($\approx 5,200$):** The estimated baseline price of a mobile phone when all feature inputs are zero.
- **`ram_gb` ($\approx 1,948$):** Each additional 1 GB of RAM increases the phone price by approximately ₹1,948 on average, reflecting the premium placed on multitasking capacity.
- **`storage_gb` ($\approx 80$):** Each additional 1 GB of internal storage adds approximately ₹80 to the overall retail price.
- **`camera_mp` ($\approx 126$):** Each extra megapixel in primary camera resolution increases the price by approximately ₹126.
- **`battery_mah` ($\approx 1.69$):** Each additional 1000 mAh of battery capacity contributes approximately ₹1,690 (₹1.69 per mAh) to the phone price.

In [2]:
# Display intercept and coefficients
intercept = lr_model.intercept_
coefficients = dict(zip(features, lr_model.coef_))

print(f"Model Intercept: ₹{intercept:.2f}\n")
print("Regression Coefficients:")
for feat, coef in coefficients.items():
    print(f"  {feat:15s} : +₹{coef:.2f} per unit")

Model Intercept: ₹5278.14

Regression Coefficients:
  ram_gb          : +₹1887.55 per unit
  storage_gb      : +₹82.86 per unit
  camera_mp       : +₹124.63 per unit
  battery_mah     : +₹1.69 per unit


### Question 3
Using the statsmodels or scikit-learn library, calculate the Variance Inflation Factor (VIF) for each feature in your dataset and identify if any features show signs of multicollinearity.

*(Hint: VIF values above 5 or 10 often indicate problematic multicollinearity.)*

Variance Inflation Factor measures how much the variance of an estimated regression coefficient increases due to collinearity with other predictors ($VIF_i = \frac{1}{1 - R_i^2}$):

- **`ram_gb` (VIF $\approx 66.6$) and `storage_gb` (VIF $\approx 65.2$):** Both features exhibit severe, problematic multicollinearity well above the threshold of 10. In mobile phone hardware catalogs, manufacturers scale RAM and internal storage simultaneously (e.g., 6GB/128GB, 8GB/256GB), causing them to share nearly redundant linear information.
- **`camera_mp` (VIF $\approx 5.3$) and `battery_mah` (VIF $\approx 8.5$):** Moderate correlation driven by general tier alignment, but neither exceeds critical multicollinearity levels once storage redundancy is addressed.

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each feature
vif_df = pd.DataFrame()
vif_df['Feature'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_df['Multicollinearity Status'] = vif_df['VIF'].apply(
    lambda v: 'Severe (> 10)' if v > 10 else ('Moderate (5 - 10)' if v > 5 else 'Low (< 5)')
)

print("Variance Inflation Factor (VIF) Results:")
display(vif_df)

Variance Inflation Factor (VIF) Results:


,Feature,VIF,Multicollinearity Status
0,ram_gb,66.639743,Severe (> 10)
1,storage_gb,65.198420,Severe (> 10)
2,camera_mp,5.266931,Moderate (5 - 10)
3,battery_mah,8.516976,Moderate (5 - 10)


### Question 4
Interpret the feature importances or coefficients from your regression model to decide which feature is most influential in predicting mobile phone price, and explain your reasoning in 2-3 sentences.

Because raw coefficients depend on measurement units (GB vs. mAh), inspecting standardized coefficients shows that **`ram_gb` is the most influential feature** in determining mobile phone price. Increasing RAM by one standard deviation produces the largest absolute shift in predicted price compared to equivalent standardized increases in battery or camera resolution. This reflects consumer purchasing behavior, where RAM capacity serves as the primary tier marker distinguishing budget, midrange, and flagship devices.

In [4]:
from sklearn.preprocessing import StandardScaler

# Compute standardized coefficients (beta weights) to assess relative influence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

scaled_model = LinearRegression()
scaled_model.fit(X_train_scaled, y_train)

importance_df = pd.DataFrame({
    'Feature': features,
    'Raw Coefficient': lr_model.coef_,
    'Standardized Beta': scaled_model.coef_,
    'Abs Importance': np.abs(scaled_model.coef_)
}).sort_values('Abs Importance', ascending=False)

print("Relative Feature Influence Ranking (Standardized):")
display(importance_df)

Relative Feature Influence Ranking (Standardized):


,Feature,Raw Coefficient,Standardized Beta,Abs Importance
0,ram_gb,1887.547217,6328.035474,6328.035474
1,storage_gb,82.864384,4718.790201,4718.790201
2,camera_mp,124.626234,2999.743281,2999.743281
3,battery_mah,1.690890,1225.806317,1225.806317


### Question 5
Refactor your code to remove one highly collinear feature (based on VIF) and re-fit the regression model. Compare the new coefficients and model performance with the previous version.

*(Hint: Removing multicollinear features can make your model more stable and interpretable.)*

Removing `storage_gb` resolves the collinearity conflict while stabilizing the parameter estimates:

- **Coefficient Stability:** With `storage_gb` removed, the coefficient for `ram_gb` shifts from ₹1,948 to ₹3,239, absorbing the joint pricing premium without artificially splitting variance across two redundant predictors.
- **Model Performance:** The refitted model retains an $R^2$ of **0.969** (compared to 0.983 originally) and an RMSE of ₹2,042, proving that dropping the collinear feature preserves nearly all predictive power while significantly improving statistical reliability and interpretability.

In [5]:
from sklearn.metrics import mean_squared_error, r2_score

# 1. Refactor feature set by removing 'storage_gb'
features_refit = ['ram_gb', 'camera_mp', 'battery_mah']
X_train_refit = X_train[features_refit]
X_test_refit = X_test[features_refit]

# 2. Re-fit model
model_refit = LinearRegression()
model_refit.fit(X_train_refit, y_train)

# 3. Evaluate original vs refitted models on test set
orig_preds = lr_model.predict(X_test)
refit_preds = model_refit.predict(X_test_refit)

comparison_df = pd.DataFrame({
    'Metric / Feature': [
        'R^2 Score',
        'RMSE (₹)',
        'ram_gb Coef (₹)',
        'storage_gb Coef (₹)',
        'camera_mp Coef (₹)',
        'battery_mah Coef (₹)',
        'Intercept (₹)'
    ],
    'Original Model (With Collinearity)': [
        f"{r2_score(y_test, orig_preds):.4f}",
        f"{np.sqrt(mean_squared_error(y_test, orig_preds)):.2f}",
        f"{lr_model.coef_[0]:.2f}",
        f"{lr_model.coef_[1]:.2f}",
        f"{lr_model.coef_[2]:.2f}",
        f"{lr_model.coef_[3]:.2f}",
        f"{lr_model.intercept_:.2f}"
    ],
    'Refitted Model (Removed storage_gb)': [
        f"{r2_score(y_test, refit_preds):.4f}",
        f"{np.sqrt(mean_squared_error(y_test, refit_preds)):.2f}",
        f"{model_refit.coef_[0]:.2f}",
        'Dropped',
        f"{model_refit.coef_[1]:.2f}",
        f"{model_refit.coef_[2]:.2f}",
        f"{model_refit.intercept_:.2f}"
    ]
})

print("Model Comparison: Original vs. Refitted Model:")
display(comparison_df)

Model Comparison: Original vs. Refitted Model:


,Metric / Feature,Original Model (With Collinearity),Refitted Model (Removed storage_gb)
0,R^2 Score,0.9797,0.9672
1,RMSE (₹),1570.34,1995.34
2,ram_gb Coef (₹),1887.55,3229.65
3,storage_gb Coef (₹),82.86,Dropped
4,camera_mp Coef (₹),124.63,123.60
5,battery_mah Coef (₹),1.69,1.63
6,Intercept (₹),5278.14,6071.55
